In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Load Model

In [3]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTIONV2.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTIONV2.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTIONV2.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_listV2.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")
print(f"Loaded features: {len(features)}")

Loaded models with calibration factor: 4.5
Loaded features: 161


### Load Player Data and Bookmaker Data

In [4]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Betr DFS,player_points,Duncan Robinson,Over,10.5,-137,2025-11-26,2025-11-26T23:26:26Z
1,Betr DFS,player_points,Duncan Robinson,Under,10.5,-137,2025-11-26,2025-11-26T23:26:26Z
2,Betr DFS,player_points,Tobias Harris,Over,10.5,-137,2025-11-26,2025-11-26T23:26:26Z
3,Betr DFS,player_points,Tobias Harris,Under,10.5,-137,2025-11-26,2025-11-26T23:26:26Z
4,Betr DFS,player_points,Cade Cunningham,Over,31.5,-137,2025-11-26,2025-11-26T23:26:26Z


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10, use_bias_adjustment=True)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 72 players...
Processing 66 players...
Generated 2031 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
1333,Julius Randle,Clint Capela,20.5,5.5,-114,-120,27.53,0.71,0.838,0.864,over,under,1,112.89,0.564,High,Low
156,LaMelo Ball,Yves Missi,20.5,6.5,-120,-130,27.56,1.98,0.829,0.835,over,under,1,103.41,0.517,High,Low
83,Jalen Brunson,Shai Gilgeous-Alexander,28.5,31.5,-130,-125,35.48,36.74,0.828,0.793,over,over,1,93.10,0.465,High,High
808,Giannis Antetokounmpo,Anthony Edwards,28.5,25.5,-112,-113,34.55,31.40,0.790,0.786,over,over,1,82.67,0.413,High,High
1094,Norman Powell,Stephen Curry,19.5,26.5,-120,-130,24.98,32.37,0.778,0.772,over,over,1,76.77,0.384,High,High
292,Sion James,Devin Booker,4.5,28.5,-110,-120,1.07,33.67,0.766,0.762,under,over,0,71.61,0.358,Low,High
1033,Tyler Herro,Kris Murray,20.5,4.5,-108,-115,23.33,1.17,0.762,0.752,over,under,0,68.38,0.342,Low,Low
391,Pascal Siakam,Zion Williamson,23.5,22.5,-110,-120,27.68,26.36,0.716,0.707,over,over,0,48.83,0.244,High,High
633,T.J. McConnell,Precious Achiuwa,9.5,7.5,-120,-120,6.39,4.67,0.704,0.705,under,under,0,45.94,0.230,Med,Med
1674,Trey Murphy III,Luke Kornet,19.5,8.5,-120,-135,23.29,5.93,0.697,0.702,over,under,0,43.86,0.219,High,Low


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 98 players...
Processing 90 players...
Generated 3784 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,RAW PREDICTION 1,RAW PREDICTION 2,BIAS ADJUSTMENT 1,BIAS ADJUSTMENT 2,CONFIDENCE 1,CONFIDENCE 2,REASON 1,REASON 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
2456,Julius Randle,Clint Capela,20.5,5.5,-114,-120,25.33,5.21,-2.2,4.5,MEDIUM,LOW,Underprediction zone,Low scorer - massive overprediction,27.53,0.71,over,under,0.838,0.864,0.7096,0.305,0.319,0.425,112.89,0.564,1,7.14,4.35,High,Low,"(13.5, 41.5)","(0.0, 9.2)",0.05,0,112.9
37,LaMelo Ball,Alex Caruso,20.5,7.0,-120,-137,25.36,6.68,-2.2,4.5,MEDIUM,LOW,Underprediction zone,Low scorer - massive overprediction,27.56,2.18,over,under,0.829,0.824,0.6691,0.283,0.246,0.360,100.72,0.504,1,7.44,5.18,High,Med,"(13.0, 42.2)","(0.0, 12.3)",0.05,0,100.7
2026,Ryan Rollins,Harrison Barnes,18.5,12.5,-115,-127,23.44,17.58,-1.1,0.1,HIGH,HIGH,Good zone with bias correction,Sweet spot - well calibrated,24.54,17.48,over,over,0.793,0.792,0.6160,0.258,0.233,0.323,84.79,0.424,1,7.38,6.12,High,High,"(10.1, 39.0)","(5.5, 29.5)",0.05,0,84.8
1460,Ja'Kobe Walter,Norman Powell,8.5,19.5,-120,-120,6.36,23.88,2.0,-1.1,MEDIUM,HIGH,Still overpredicting,Good zone with bias correction,4.36,24.98,under,over,0.780,0.778,0.5948,0.234,0.233,0.303,78.44,0.392,1,5.37,7.15,Med,High,"(0.0, 14.9)","(11.0, 39.0)",0.05,0,78.4
1765,Ben Sheppard,Will Richard,6.5,7.0,-137,-137,7.04,7.54,4.5,4.5,LOW,LOW,Low scorer - massive overprediction,Low scorer - massive overprediction,2.54,3.04,under,under,0.765,0.772,0.5789,0.187,0.194,0.251,73.66,0.368,0,5.47,5.33,Med,Med,"(0.0, 13.3)","(0.0, 13.5)",0.05,0,73.7
903,Bennedict Mathurin,Tyler Herro,20.5,19.5,-145,-125,22.80,21.13,-2.2,-1.1,MEDIUM,HIGH,Underprediction zone,Good zone with bias correction,25.00,22.23,over,over,0.751,0.754,0.5546,0.159,0.198,0.232,66.37,0.332,0,6.65,3.98,High,Low,"(12.0, 38.0)","(14.4, 30.0)",0.05,0,66.4
2108,Andrew Wiggins,Saddiq Bey,13.5,12.5,-145,-143,17.76,17.05,0.1,0.1,HIGH,HIGH,Sweet spot - well calibrated,Sweet spot - well calibrated,17.66,16.95,over,over,0.720,0.727,0.5128,0.128,0.138,0.172,53.84,0.269,1,7.14,7.37,High,High,"(3.7, 31.6)","(2.5, 31.4)",0.05,0,53.8
852,Pascal Siakam,Jeremiah Fears,23.5,14.5,-110,-135,25.48,18.50,-2.2,0.1,MEDIUM,HIGH,Underprediction zone,Sweet spot - well calibrated,27.68,18.40,over,over,0.716,0.711,0.4989,0.192,0.137,0.204,49.68,0.248,0,7.34,7.00,High,High,"(13.3, 42.1)","(4.7, 32.1)",0.05,0,49.7
2892,Zion Williamson,Precious Achiuwa,22.5,7.5,-120,-120,24.16,6.67,-2.2,2.0,MEDIUM,MEDIUM,Underprediction zone,Still overpredicting,26.36,4.67,over,under,0.707,0.705,0.4890,0.162,0.160,0.197,46.70,0.233,0,7.08,5.24,High,Med,"(12.5, 40.2)","(0.0, 14.9)",0.05,0,46.7
1386,T.J. McConnell,Luke Kornet,9.5,8.5,-120,-135,8.39,7.93,2.0,2.0,MEDIUM,MEDIUM,Still overpredicting,Still overpredicting,6.39,5.93,under,under,0.704,0.702,0.4843,0.158,0.128,0.178,45.30,0.227,0,5.81,4.84,Med,Low,"(0.0, 17.8)","(0.0, 15.4)",0.05,0,45.3


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 61 players...
Processing 56 players...
Generated 23276 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
18024,Julius Randle,Yves Missi,Clint Capela,20.5,6.5,5.5,27.53,1.98,0.71,0.838,0.835,0.864,over,under,under,1,226.50,0.453,High,Low,Low
62,LaMelo Ball,Pascal Siakam,Norman Powell,20.5,23.5,19.5,27.56,27.68,24.98,0.829,0.716,0.778,over,over,over,1,149.29,0.299,High,High,High
14566,Tyler Herro,Zion Williamson,Precious Achiuwa,20.5,22.5,7.5,23.33,26.36,4.67,0.762,0.707,0.705,over,over,under,0,105.26,0.211,Low,High,Med
8943,T.J. McConnell,Trey Murphy III,Luke Kornet,9.5,19.5,8.5,6.39,23.29,5.93,0.704,0.697,0.702,under,over,under,0,85.93,0.172,Med,High,Low
2608,Josh Hart,Josh Okogie,Julian Champagnie,13.5,9.5,11.5,10.42,6.64,14.64,0.688,0.694,0.682,under,under,over,0,75.71,0.151,High,Med,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 98 players...
Processing 90 players...
Generated 98658 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
1952,LaMelo Ball,Julius Randle,Clint Capela,20.5,20.5,5.5,27.56,27.53,0.71,0.829,0.838,0.864,over,over,under,1,224.01,0.448,High,High,Low
67893,Ryan Rollins,Alex Caruso,Harrison Barnes,18.5,7.0,12.5,24.54,2.18,17.48,0.793,0.824,0.792,over,under,over,1,179.66,0.359,High,Med,High
52224,Ja'Kobe Walter,Norman Powell,Will Richard,8.5,19.5,7.0,4.36,24.98,3.04,0.780,0.778,0.772,under,over,under,0,152.92,0.306,Med,High,Med
60256,Ben Sheppard,Tyler Herro,Saddiq Bey,6.5,19.5,12.5,2.54,22.23,16.95,0.765,0.754,0.727,under,over,over,0,126.46,0.253,Med,Low,High
34608,Bennedict Mathurin,Andrew Wiggins,Jeremiah Fears,20.5,13.5,14.5,25.00,17.66,18.40,0.751,0.720,0.711,over,over,over,0,107.61,0.215,High,High,High


In [9]:
df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)
len(df)

105